# GenSticker — SDXL + InstantID + chibi sticker

Notebook mới sau khi audit các thử nghiệm cũ trong `GenSticker-AI`.

**Kết luận audit:** notebook cũ không dùng InstantID/InsightFace; đa số output là `cpu_cartoon_fallback`; `img2img strength=0.88` phá danh tính; prompt vượt giới hạn CLIP; mask của ảnh gốc bị tái sử dụng cho hình đã đổi hình học; và đường dẫn model không khớp cấu trúc Drive hiện tại.

Notebook này là pipeline **một ảnh — một người — một sticker**. Ảnh đầu vào phải có đúng một khuôn mặt; trường hợp có từ hai người trở lên sẽ dừng với thông báo chưa hỗ trợ. Ca kiểm thử đầu tiên dùng `inputs/pilot.jpg`, giữ danh tính bằng InstantID + InsightFace, giữ cấu trúc bằng IdentityNet ControlNet, sau đó tách nền bằng BiRefNet và thêm viền trắng.

In [ ]:
# CELL 1 — Cài dependency, mount Drive và xác nhận GPU
%pip install -q -U "diffusers==0.35.1" "transformers>=4.44,<5" "accelerate>=0.33" "peft>=0.12" "safetensors>=0.4.5" "huggingface_hub>=0.25" "insightface==0.7.3" "onnxruntime-gpu==1.20.1" "kornia>=0.7.3" gdown opencv-python-headless

from pathlib import Path
from google.colab import drive
import torch

drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive/GenSticker-AI')
INPUTS = ROOT / 'inputs'
OUTPUTS = ROOT / 'outputs'
MODELS = ROOT / 'models'
SDXL_DIR = MODELS / 'sdxl-base'
LORA_FILE = MODELS / 'lora' / 'StickersRedmond.safetensors'
BIREF_DIR = MODELS / 'birefnet'
INSTANT_DIR = MODELS / 'instantid'
FACE_ROOT = MODELS / 'insightface'

assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trước khi chạy notebook.'
for path in [ROOT, INPUTS, OUTPUTS, SDXL_DIR, LORA_FILE, BIREF_DIR]:
    assert path.exists(), f'MISSING: {path}'

print('GPU:', torch.cuda.get_device_name(0))
print('ROOT_OK:', ROOT)


In [ ]:
# CELL 2 — Tải đúng checkpoint InstantID và antelopev2 vào Drive (chỉ tải lần đầu)
import os, shutil, subprocess, sys
from huggingface_hub import hf_hub_download
import gdown

REPO = Path('/content/InstantID')
if not (REPO / 'pipeline_stable_diffusion_xl_instantid.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/instantX-research/InstantID.git', str(REPO)], check=True)

INSTANT_DIR.mkdir(parents=True, exist_ok=True)
for filename in [
    'ControlNetModel/config.json',
    'ControlNetModel/diffusion_pytorch_model.safetensors',
    'ip-adapter.bin',
]:
    target = INSTANT_DIR / filename
    if not target.exists():
        hf_hub_download(repo_id='InstantX/InstantID', filename=filename, local_dir=str(INSTANT_DIR))

face_models = FACE_ROOT / 'models'
antelope_dir = face_models / 'antelopev2'
if not (antelope_dir / 'glintr100.onnx').exists():
    face_models.mkdir(parents=True, exist_ok=True)
    antelope_zip = FACE_ROOT / 'antelopev2.zip'
    if not antelope_zip.exists():
        gdown.download(id='18wEUfMNohBJ4K3Ly5wpTejPfDzp-8fI8', output=str(antelope_zip), quiet=False)
    shutil.unpack_archive(str(antelope_zip), str(face_models))

required = [
    INSTANT_DIR / 'ip-adapter.bin',
    INSTANT_DIR / 'ControlNetModel' / 'diffusion_pytorch_model.safetensors',
    antelope_dir / 'glintr100.onnx',
    antelope_dir / 'scrfd_10g_bnkps.onnx',
]
for path in required:
    assert path.exists(), f'Download chưa hoàn chỉnh: {path}'
print('INSTANTID_ASSETS_OK')


In [ ]:
# CELL 3 — Load InsightFace, IdentityNet và SDXL InstantID
import gc, json, math, sys
import cv2
import numpy as np
from PIL import Image, ImageFilter
from insightface.app import FaceAnalysis
from diffusers import ControlNetModel, DPMSolverMultistepScheduler

sys.path.insert(0, str(REPO))
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps

face_app = FaceAnalysis(
    name='antelopev2',
    root=str(FACE_ROOT),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
face_app.prepare(ctx_id=0, det_size=(640, 640))

controlnet = ControlNetModel.from_pretrained(
    str(INSTANT_DIR / 'ControlNetModel'),
    torch_dtype=torch.float16,
    use_safetensors=True,
    local_files_only=True,
)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    str(SDXL_DIR),
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
    local_files_only=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type='sde-dpmsolver++', use_karras_sigmas=True)
pipe.load_ip_adapter_instantid(str(INSTANT_DIR / 'ip-adapter.bin'))
pipe.set_ip_adapter_scale(0.78)

STYLE_BACKEND = 'InstantID + SDXL base'
try:
    pipe.load_lora_weights(str(LORA_FILE.parent), weight_name=LORA_FILE.name)
    pipe.fuse_lora(lora_scale=0.65)
    STYLE_BACKEND += ' + StickersRedmond LoRA'
except Exception as exc:
    print('LoRA không tương thích, tiếp tục bằng prompt chibi:', type(exc).__name__)

pipe.enable_model_cpu_offload()
pipe.enable_vae_tiling()
print('PIPELINE_READY:', STYLE_BACKEND)


In [ ]:
# CELL 4 — Tạo sticker từ một ảnh có đúng một người
from datetime import datetime
from IPython.display import display
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

INPUT_FILE = 'pilot.jpg'
INPUT_PATH = INPUTS / INPUT_FILE
source = Image.open(INPUT_PATH).convert('RGB')
source_bgr = cv2.cvtColor(np.asarray(source), cv2.COLOR_RGB2BGR)
detected = face_app.get(source_bgr)

if len(detected) == 0:
    raise ValueError('Không phát hiện khuôn mặt. Hãy dùng ảnh chân dung rõ mặt.')
if len(detected) > 1:
    raise ValueError(f'Pipeline hiện chỉ hỗ trợ đúng 1 người; ảnh này có {len(detected)} khuôn mặt.')
original_face = detected[0]

def crop_face_context(image, face, scale=3.0, out_size=1024):
    x0, y0, x1, y1 = map(float, face.bbox)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(x1 - x0, y1 - y0) * scale
    box = (int(cx - side/2), int(cy - side*0.43), int(cx + side/2), int(cy + side*0.57))
    crop = Image.new('RGB', (max(1, box[2]-box[0]), max(1, box[3]-box[1])), 'white')
    valid = (max(0, box[0]), max(0, box[1]), min(image.width, box[2]), min(image.height, box[3]))
    patch = image.crop(valid)
    crop.paste(patch, (valid[0]-box[0], valid[1]-box[1]))
    return crop.resize((out_size, out_size), Image.Resampling.LANCZOS)

reference = crop_face_context(source, original_face)
reference_faces = face_app.get(cv2.cvtColor(np.asarray(reference), cv2.COLOR_RGB2BGR))
if len(reference_faces) != 1:
    raise ValueError(f'Ảnh sau khi căn khung phải có đúng 1 khuôn mặt; phát hiện {len(reference_faces)}.')
face = reference_faces[0]
keypoints = draw_kps(reference, face.kps)

prompt = (
    'one super-deformed cute 2D chibi character of the same person, '
    'recognizable facial identity and hairstyle, very large rounded head, oversized sparkling eyes, '
    'tiny simplified shoulders and body, kawaii anime proportions, clean line art, bold outline, '
    'smooth cel shading, centered, isolated on plain white background, no text, no typography, no caption'
)
negative = (
    'multiple people, duplicate person, photorealistic, realistic skin, realistic proportions, '
    'fashion portrait, cleavage, long body, small eyes, deformed face, extra limbs, bad hands, '
    'text, watermark, logo, blurry, low quality'
)

pipe.set_ip_adapter_scale(0.68)
raw = pipe(
    prompt=prompt,
    negative_prompt=negative,
    image_embeds=face.embedding,
    image=keypoints,
    width=1024,
    height=1024,
    num_inference_steps=32,
    guidance_scale=5.0,
    controlnet_conditioning_scale=0.72,
    generator=torch.Generator(device='cpu').manual_seed(20260731),
).images[0]
display(reference.resize((320, 320)), raw.resize((512, 512)))

del pipe, controlnet
gc.collect(); torch.cuda.empty_cache()

biref = AutoModelForImageSegmentation.from_pretrained(
    str(BIREF_DIR), trust_remote_code=True, local_files_only=True
).to('cuda').eval()
prep = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def sticker_rgba(image):
    x = prep(image).unsqueeze(0).to('cuda')
    with torch.inference_mode():
        pred = biref(x)[-1].sigmoid().float().cpu()[0].squeeze().numpy()
    mask = Image.fromarray(np.uint8(np.clip(pred, 0, 1) * 255)).resize(image.size, Image.Resampling.LANCZOS)
    mask = mask.filter(ImageFilter.GaussianBlur(0.8))
    outline = mask.filter(ImageFilter.MaxFilter(31))
    white = Image.new('RGBA', image.size, (255, 255, 255, 0)); white.putalpha(outline)
    subject = image.convert('RGBA'); subject.putalpha(mask)
    return Image.alpha_composite(white, subject)

sticker = sticker_rgba(raw)
run_id = 'instantid-single-' + datetime.now().strftime('%Y%m%d-%H%M%S')
out_dir = OUTPUTS / run_id
out_dir.mkdir(parents=True, exist_ok=False)

reference.save(out_dir / 'reference.png')
raw.save(out_dir / 'raw.png')
sticker.save(out_dir / 'sticker.png', optimize=True)
display(sticker.resize((512, 512)))

manifest = {
    'input': str(INPUT_PATH),
    'people_supported': 1,
    'faces_detected': len(detected),
    'pipeline': STYLE_BACKEND,
    'face_model': 'InsightFace antelopev2',
    'ip_adapter_scale': 0.68,
    'controlnet_conditioning_scale': 0.72,
    'steps': 32,
    'guidance_scale': 5.0,
    'prompt': prompt,
    'output': str(out_dir / 'sticker.png'),
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = shutil.make_archive(str(out_dir), 'zip', root_dir=out_dir)
print('FACES_DETECTED:', len(detected))
print('OUTPUT_DIR:', out_dir)
print('OUTPUT_ZIP:', zip_path)
